# 00 はじめに：環境構築と動作確認 / Getting started: setup and check

このチュートリアルでは，数 nm の合金ナノ粒子（FePt）を例に，原子分解能電子線トモグラフィー（AET）の基本的な流れを自分の PC で一通り体験します．

This tutorial walks you through the basic workflow of atomic electron tomography (AET) on your own computer, using an alloy (FePt) nanoparticle a few nanometres across as the example.

| ノートブック / Notebook | 内容 / Contents |
|---|---|
| `00_setup.ipynb` | 環境構築と動作確認（このページ）<br>Setup and check (this page) |
| `01_model_and_tilt_series.ipynb` | 原子モデルと，ノイズを含む傾斜シリーズを作る<br>Build an atomic model and a noisy tilt series |
| `02_genfire.ipynb` | GENFIRE（フーリエ空間の反復再構成）<br>GENFIRE (Fourier-space iterative reconstruction) |
| `03_resire.ipynb` | RESIRE（実空間の反復再構成）<br>RESIRE (real-space iterative reconstruction) |
| `04_atom_tracing.ipynb` | 原子の抽出，元素の分類，正解との比較<br>Atom tracing, species classification, comparison with the ground truth |

順番に実行してください．01 で作ったデータを 02 以降で読み込みます．各ノートブックは 10〜30 秒程度で実行できます（ノート PC での目安）．

Run the notebooks in order: 02 onwards read the data made in 01. Each notebook takes about 10–30 seconds on a laptop.

---

## 1. 環境構築のコマンド / Setup commands

Python の環境は [uv](https://docs.astral.sh/uv/) で作ります．uv は Python 本体と必要なパッケージをまとめて用意してくれるツールで，あらかじめ Python をインストールしておく必要はありません．コマンドは**ターミナル**（Windows では「PowerShell」，macOS / Linux では「ターミナル」）で実行します．

The Python environment is set up with [uv](https://docs.astral.sh/uv/). uv installs Python itself together with all required packages, so you do not need to install Python beforehand. Run the commands below in a **terminal** (PowerShell on Windows, Terminal on macOS / Linux).

### (1) uv のインストール（最初の 1 回だけ） / Install uv (once)

Windows (PowerShell):

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

macOS / Linux:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

インストールが終わったらターミナルを開き直し，`uv --version` でバージョンが表示されることを確認します．

After the installation, open a new terminal and check that `uv --version` prints a version number.

### (2) ダウンロードと環境構築（最初の 1 回だけ） / Download and set up (once)

```bash
git clone https://github.com/ty-URC/AET-tutorials.git
cd AET-tutorials
uv sync
```

`uv sync` で，Python 3.12 と必要なパッケージ（NumPy, SciPy, Matplotlib, pyFFTW, JupyterLab など）が `AET-tutorials/.venv` フォルダにインストールされます．パッケージのバージョンは `uv.lock` に記録されているので，誰が実行しても同じ環境になります．

`uv sync` installs Python 3.12 and the required packages (NumPy, SciPy, Matplotlib, pyFFTW, JupyterLab, ...) into the `AET-tutorials/.venv` folder. The package versions are pinned in `uv.lock`, so everyone gets the same environment.

### (3) JupyterLab の起動（毎回） / Start JupyterLab (every time)

```bash
cd AET-tutorials
uv run jupyter lab
```

- `git` がない場合は，GitHub のページの「Code」→「Download ZIP」でダウンロードして展開し，そのフォルダで `cd` 以降を実行します．
  Without `git`, use "Code" → "Download ZIP" on the GitHub page, unzip it, and run the commands from `cd` onwards in that folder.
- `uv run jupyter lab` を実行するとブラウザが開きます．`notebooks` フォルダを開き，`00_setup.ipynb` から順に開いてください．
  `uv run jupyter lab` opens a browser. Open the `notebooks` folder and work through the notebooks in order, starting from `00_setup.ipynb`.
- セルは `Shift + Enter` で 1 つずつ実行できます．「Run」→「Run All Cells」で全部を実行することもできます．
  Run cells one by one with `Shift + Enter`, or all at once with "Run" → "Run All Cells".
- 終えるときは，ブラウザを閉じてからターミナルで `Ctrl + C` を押します．
  To quit, close the browser and press `Ctrl + C` in the terminal.
- 新しい版に更新するときは，`AET-tutorials` フォルダで `git pull` → `uv sync` を実行します．
  To update to a new version, run `git pull` and then `uv sync` in the `AET-tutorials` folder.

## 2. 動作確認 / Check the installation

下のセルを実行して，必要なパッケージが読み込めるか確かめます．エラーが出た場合は，`AET-tutorials` フォルダで `uv sync` を実行してから，`uv run jupyter lab` で JupyterLab を起動しているか確認してください．

Run the cells below to check that the packages can be imported. If you get an error, run `uv sync` in the `AET-tutorials` folder and make sure JupyterLab was started with `uv run jupyter lab`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent          # リポジトリの最上位 / repository root
sys.path.insert(0, str(ROOT / "src"))     # src/ のパッケージを使う / use the packages in src/
DATA = ROOT / "data"                      # データの保存先 / where data are saved
DATA.mkdir(exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 100

In [2]:
import platform, os
import scipy, matplotlib
print("Python     :", platform.python_version())
print("NumPy      :", np.__version__)
print("SciPy      :", scipy.__version__)
print("Matplotlib :", matplotlib.__version__)
print("CPU cores  :", os.cpu_count())

import genfire_cpu, resire_cpu, aet_tools
print("genfire_cpu:", genfire_cpu.__version__, "(FFT: %s)" % ("pyFFTW" if genfire_cpu.HAVE_PYFFTW else "SciPy"))
print("resire_cpu :", resire_cpu.__version__)
print("Ready.")

Python     : 3.12.16
NumPy      : 2.5.4
SciPy      : 1.18.1
Matplotlib : 3.11.2
CPU cores  : 22


genfire_cpu: 0.1.0 (FFT: pyFFTW)
resire_cpu : 0.1.0
Ready.


`FFT: SciPy` と表示されても問題ありません．pyFFTW があると GENFIRE が少し速くなります．

`FFT: SciPy` is fine too; pyFFTW only makes GENFIRE a little faster.

## 3. リポジトリの構成 / Repository layout

```
AET-tutorials/
├── notebooks/        ノートブック / notebooks
├── src/
│   ├── genfire_cpu/  GENFIRE の CPU 実装 / CPU implementation of GENFIRE
│   ├── resire_cpu/   RESIRE の CPU 実装 / CPU implementation of RESIRE
│   └── aet_tools/    モデル・ノイズ・原子の抽出・表示 / model, noise, atom tracing, plotting
├── data/             ノートブックが作るデータ / data written by the notebooks
├── pyproject.toml    必要なパッケージ / required packages (read by uv)
└── uv.lock           パッケージのバージョン / pinned package versions (written by uv)
```

## 4. 座標の約束 / Coordinate conventions

- 三次元像は `volume[x, y, z]` の配列です．**y 軸が傾斜軸**で，傾斜 0° のとき電子線は z 方向に進みます．
  A volume is an array `volume[x, y, z]`. **The y axis is the tilt axis**, and at 0° tilt the beam travels along z.
- 投影像は `projections[x, y, 傾斜番号]` の配列です．
  Projections are stored as `projections[x, y, tilt index]`.
- 画像は横軸を x（傾斜軸に垂直），縦軸を y（傾斜軸）として表示します．
  Images are shown with x (perpendicular to the tilt axis) horizontal and y (the tilt axis) vertical.

次は `01_model_and_tilt_series.ipynb` に進みます． / Next: `01_model_and_tilt_series.ipynb`.